In [ ]:
# Import the clean_sac_file function from the previous example
from obspy import read, Trace, Stream
from obspy.signal.filter import bandpass
import numpy as np

def clean_sac_file(input_file, output_file, freq_min = 2.0, freq_max = 4.3, time_pad = 10.001):
    """
    Applies a bandpass filter to a SAC file to clean noise.
    Inputs:
    - input_file: path to the input SAC file
    - output_file: path to the output SAC file
    - freq_min: minimum frequency of the passband in Hz
    - freq_max: maximum frequency of the passband in Hz
    """
    # Read in the SAC file as an ObsPy Trace object
    trace = read(input_file)[0]
    # trace.plot()
    
    # Apply a bandpass filter to the trace
    trace_filtered = trace.copy()
    t = trace_filtered.stats.starttime

    if t + time_pad > trace_filtered.stats.endtime:
        return None
    
    trace_filtered = trace_filtered.trim(t + time_pad, trace_filtered.stats.endtime) 
    # trace_filtered = trace_filtered.trim(t, trace_filtered.stats.endtime)
    # trace_filtered.data = np.concatenate((np.zeros(750), trace_filtered.data))
    
    trace_filtered.data = bandpass(trace_filtered.data, freqmin=freq_min, freqmax=freq_max, df=trace_filtered.stats.sampling_rate)
    # trace_filtered = trace_filtered.trim(t + time_pad, trace_filtered.stats.endtime)
    # trace_filtered.plot()

    return trace_filtered

In [ ]:
import os
import shutil
from obspy import read, Trace, Stream

os.makedirs("class0") # yok
os.makedirs("class0/clean") 
os.makedirs("class0/original")

os.makedirs("class1") # 1 seviye 250
os.makedirs("class1/clean")
os.makedirs("class1/original")

os.makedirs("class2") # 2 seviye 500
os.makedirs("class2/clean")
os.makedirs("class2/original")

os.makedirs("class3") # 3 seviye 1000
os.makedirs("class3/clean")
os.makedirs("class3/original")

os.makedirs("class4") # 4 seviye 2000
os.makedirs("class4/clean")
os.makedirs("class4/original")

os.makedirs("class5") # 5 seviye 4000
os.makedirs("class5/clean")
os.makedirs("class5/original")

os.makedirs("class6") # 6 seviye 8000   
os.makedirs("class6/clean")
os.makedirs("class6/original")

os.makedirs("class7") # 6 seviye 16000   
os.makedirs("class7/clean")
os.makedirs("class7/original")

os.makedirs("class8") # 6 seviye 32000   
os.makedirs("class8/clean")
os.makedirs("class8/original")

# Define the directory path
dir_path = './'
orderdata = '201907_'

c = 0

clean_files = []

for i, folder_month in enumerate(os.listdir(dir_path)):

    if not os.path.isfile(dir_path + folder_month):
        # print(folder_month)

        if folder_month.__contains__("class"):
            continue

        for j, eartq in enumerate(os.listdir(dir_path + folder_month)):

            if not os.path.join(dir_path, folder_month, eartq).format().__contains__("BHZ"):
                continue

            # print(f"    {eartq}")
            
            date = str(i) + "_" + str(j) + "_"
            
            # c = c + 1
            # continue

            # Define the input and output file paths
            input_file = dir_path + folder_month + '/' + eartq
            output_file = orderdata + date + eartq.replace('(', '').replace(')', '').replace('=', '').replace('KO', 'SAC')


            file_name = dir_path + folder_month + '/' + eartq


            if clean_files.__contains__(file_name):
                continue
            
            print(input_file)
            trace_filtered = clean_sac_file(input_file, output_file)
            if trace_filtered is None:
                continue

            clean_files.append(file_name)

            # if trace_filtered.stast.endtime - trace_filtered.stats.starttime < 15:
            #     continue

            trace_filtered = trace_filtered[300:]
            if len(trace_filtered.data) < 8000:
                continue

            # crop 8000 sample in the middle
            lengthOfArray = len(trace_filtered.data)

            max_val = max(trace_filtered.data)
            max_value_index = np.argmax(trace_filtered.data)

            min_val = min(trace_filtered.data)
            min_value_index = np.argmin(trace_filtered.data)

            write_signal = read(input_file)[0]
            max_orig_val = max(write_signal.data)
            min_orig_val = min(write_signal.data)

            add_num = 0
            # according to max and min value values place the array in the middle by 0 value
            # if max_orig_val < 0 and min_orig_val < 0:
            #     diff = abs(max_orig_val) - abs(min_orig_val)
            #     add_num = diff
            # elif max_orig_val > 0 and min_orig_val > 0:
            #     diff = abs(max_orig_val) - abs(min_orig_val)
            #     add_num = -diff

            a = 0
            res_signal = Trace()
            if lengthOfArray - max_value_index < 4000:
                a = 1
                rightArr = trace_filtered.data[max_value_index:]
                rest_length = 8000 - len(rightArr)
                leftArr = trace_filtered.data[max_value_index - rest_length: max_value_index]
                res_signal.data = np.concatenate((leftArr, rightArr), axis=0)

                write_signal.data = write_signal.data[max_value_index - rest_length:] + add_num

            elif max_value_index < 4000:
                a = 2
                leftArr = trace_filtered.data[0: max_value_index]
                rest_length = 8000 - len(leftArr)
                rightArr = trace_filtered.data[max_value_index: max_value_index + rest_length]
                res_signal.data = np.concatenate((leftArr, rightArr), axis=0)

                write_signal.data = write_signal.data[0: max_value_index + rest_length] + add_num

            else:
                a = 3
                res_signal.data = np.array(trace_filtered.data[max_value_index - 4000: max_value_index + 4000])

                write_signal.data = write_signal.data[max_value_index - 4000: max_value_index + 4000] + add_num

            if len(res_signal.data) < 8000:
                print("error" + (str(a)))
                continue

            x3 = 1
            x2 = 2
            x1 = 8

            print(f"Len : {len(write_signal.data)}")

            random_number = np.random.randint(0, x1 + x2 + x3)
            
            if random_number < x1:
                max_val = max_val * 3
            elif random_number < x1 + x2:
                max_val = max_val * 2
            else:
                max_val = max_val * 1


            if max_val < 250:
                res_signal.write('./class0/clean/' + output_file, format='SAC')
                write_signal.write('./class0/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            elif max_val < 500:
                res_signal.write('./class1/clean/' + output_file, format='SAC')
                write_signal.write('./class1/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            elif max_val < 1000:
                res_signal.write('./class2/clean/' + output_file, format='SAC')
                write_signal.write('./class2/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            elif max_val < 2000:
                res_signal.write('./class3/clean/' + output_file, format='SAC')
                write_signal.write('./class3/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            elif max_val < 4000:
                res_signal.write('./class4/clean/' + output_file, format='SAC')
                write_signal.write('./class4/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            elif max_val < 8000:
                res_signal.write('./class5/clean/' + output_file, format='SAC')
                write_signal.write('./class5/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            elif max_val < 16000:
                res_signal.write('./class6/clean/' + output_file, format='SAC')
                write_signal.write('./class6/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            elif max_val < 32000:
                res_signal.write('./class7/clean/' + output_file, format='SAC')
                write_signal.write('./class7/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')
            else:
                res_signal.write('./class8/clean/' + output_file, format='SAC')
                write_signal.write('./class8/original/' + orderdata + date + "_" + eartq + '_original.sac', format='SAC')           


            # if max_val < 250:
            #     res_signal.write('./class0/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class0/' + orderdata + date + "_" + eartq + '_original.sac')
            # elif max_val < 500:
            #     res_signal.write('./class1/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class1/' + orderdata + date + "_" + eartq + '_original.sac')
            # elif max_val < 1000:
            #     res_signal.write('./class2/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class2/' + orderdata + date + "_" + eartq + '_original.sac')
            # elif max_val < 2000:
            #     res_signal.write('./class3/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class3/' + orderdata + date + "_" + eartq + '_original.sac')
            # elif max_val < 4000:
            #     res_signal.write('./class4/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class4/' + orderdata + date + "_" + eartq + '_original.sac')
            # elif max_val < 8000:
            #     res_signal.write('./class5/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class5/' + orderdata + date + "_" + eartq + '_original.sac')
            # elif max_val < 16000:
            #     res_signal.write('./class6/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class6/' + orderdata + date + "_" + eartq + '_original.sac')
            # elif max_val < 32000:
            #     res_signal.write('./class7/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class7/' + orderdata + date + "_" + eartq + '_original.sac')
            # else:
            #     res_signal.write('./class8/' + output_file, format='SAC')
            #     shutil.copy(input_file, './class8/' + orderdata + date + "_" + eartq + '_original.sac')


print(c)

In [ ]:
import os
import shutil
from obspy import read, Trace, Stream

os.makedirs("class0") # yok
os.makedirs("class1") # 1 seviye 250
os.makedirs("class2") # 2 seviye 500
os.makedirs("class3") # 3 seviye 1000
os.makedirs("class4") # 4 seviye 2000
os.makedirs("class5") # 5 seviye 4000
os.makedirs("class6") # 6 seviye 8000   
os.makedirs("class7") # 6 seviye 16000   
os.makedirs("class8") # 6 seviye 32000   


# Define the directory path
dir_path = './'
orderdata = '201907_'

c = 0

clean_files = []

for i, folder_month in enumerate(os.listdir(dir_path)):

    if not os.path.isfile(dir_path + folder_month):
        # print(folder_month)

        for j, eartq in enumerate(os.listdir(dir_path + folder_month)):

            if not os.path.join(dir_path, folder_month, eartq).format().__contains__("BHZ"):
                continue

            # print(f"    {eartq}")
            
            date = str(i) + "_" + str(j) + "_"
            
            # c = c + 1
            # continue

            # Define the input and output file paths
            input_file = dir_path + folder_month + '/' + eartq
            output_file = orderdata + date + eartq.replace('(', '').replace(')', '').replace('=', '').replace('KO', 'SAC')


            file_name = dir_path + folder_month + '/' + eartq


            if clean_files.__contains__(file_name):
                continue

            trace_filtered = clean_sac_file(input_file, output_file)
            if trace_filtered is None:
                continue

            clean_files.append(file_name)

            # if trace_filtered.stast.endtime - trace_filtered.stats.starttime < 15:
            #     continue

            min_val = min(trace_filtered.data[150:])
            max_val = max(trace_filtered.data[300:])

            max_value_index = np.argmax(trace_filtered.data[300:])
            # get the 1000 padding centered max index
            trace_filtered.data = trace_filtered.data[max_value_index - 2000: max_value_index + 2000]
           
            # print(f"min: {min_val} max: {max_val}")
            # print(file_name)
            # if max_val > 16000:
            #     trace_filtered.write('./class6/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class6/' + eartq + '_original.sac')

            if max_val < 250:
                trace_filtered.write('./class0/' + output_file, format='SAC')
                # shutil.copy(file_name, './class0/' + eartq + '_original.sac')
            elif max_val < 500:
                trace_filtered.write('./class1/' + output_file, format='SAC')
                # shutil.copy(file_name, './class1/' + eartq + '_original.sac')
            elif max_val < 1000:
                trace_filtered.write('./class2/' + output_file, format='SAC')
                # shutil.copy(file_name, './class2/' + eartq + '_original.sac')
            elif max_val < 2000:
                trace_filtered.write('./class3/' + output_file, format='SAC')
                # shutil.copy(file_name, './class3/' + eartq + '_original.sac')
            elif max_val < 4000:
                trace_filtered.write('./class4/' + output_file, format='SAC')
                # shutil.copy(file_name, './class4/' + eartq + '_original.sac')
            elif max_val < 8000:
                trace_filtered.write('./class5/' + output_file, format='SAC')
                # shutil.copy(file_name, './class5/' + eartq + '_original.sac')
            elif max_val < 16000:
                trace_filtered.write('./class6/' + output_file, format='SAC')
                # shutil.copy(file_name, './class6/' + eartq + '_original.sac')
            elif max_val < 32000:
                trace_filtered.write('./class7/' + output_file, format='SAC')
                # shutil.copy(file_name, './class7/' + eartq + '_original.sac')
            else:
                trace_filtered.write('./class8/' + output_file, format='SAC')
                # shutil.copy(file_name, './class8/' + eartq + '_original.sac')



            # if max_val < 500:
            #     trace_filtered.write('./class0/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class0/' + eartq + '_original.sac')
            # elif max_val < 1000:
            #     trace_filtered.write('./class1/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class1/' + eartq + '_original.sac')
            # elif max_val < 2000:
            #     trace_filtered.write('./class2/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class2/' + eartq + '_original.sac')
            # elif max_val < 4000:
            #     trace_filtered.write('./class3/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class3/' + eartq + '_original.sac')
            # elif max_val < 8000:
            #     trace_filtered.write('./class4/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class4/' + eartq + '_original.sac')
            # elif max_val < 16000:
            #     trace_filtered.write('./class5/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class5/' + eartq + '_original.sac')
            # else:
            #     trace_filtered.write('./class6/' + output_file, format='SAC')
            #     # shutil.copy(file_name, './class6/' + eartq + '_original.sac')

print(c)

In [ ]:

#  read sac file

from obspy import read, Trace, Stream
import os

path = "./"
folder_classes = ["class8", "class7", "class6", "class5", "class4", "class3", "class2", "class1", "class0"]

for folder_class in folder_classes:
    files = []
    for j, file in enumerate(os.listdir(path= path + folder_class)):
        files.append(file)

    files = sorted(files)

    for i, file in enumerate(files):
        print(file)
        trace = read(path + folder_class + "/" + file)[0]
        print(len(trace.data))

        if len(trace.data) == 4000:
            print(trace.stats)
            # trace.plot()
            # ix = input("Press Enter to continue...")
            # if ix == 'q':
            #     break
        else:
            os.remove(path + folder_class + "/" + file)


In [ ]:


from obspy import read, Trace, Stream
import os

path = "./"

folder_class = "class5"

files = []
for j, file in enumerate(os.listdir(path= path + folder_class)):
    files.append(file)

files = sorted(files)

for i, file in enumerate(files):
    print(file)
    trace = read(path + folder_class + "/" + file)[0]
    print(len(trace.data))

    if len(trace.data) == 4000:
        # print(trace.stats)
        trace.plot()
        # ix = input("Press Enter to continue...")
        # if ix == 'q':
        #     break
        
    else:
        os.remove(path + folder_class + "/" + file)


In [ ]:
from obspy import read, Trace, Stream

path = "./class8/201905_3_0_SIMA.BHZ.SAC"

trace = read(path)[0]

print(trace.stats.endtime - trace.stats.starttime)
# trace.plot()

filtered = clean_sac_file(path, "")
# filtered.plot()


In [ ]:
import os

# Define the directory path
dir_path = './'

for folder_month in os.listdir(dir_path):

    if not os.path.isfile(dir_path + folder_month):
        print(folder_month)

        for eartq in os.listdir(dir_path + folder_month):
            print(f"    {eartq}")
            
            for sac_file in os.listdir(dir_path + folder_month + '/' + eartq):
                print(f"        {sac_file}")
                # Define the input and output file paths
                input_file = dir_path + folder_month + '/' + eartq + '/' + sac_file
                output_file = dir_path + folder_month + '/' + eartq + '/' + sac_file[:-4] + '_filtered.sac'

                # Apply the clean_sac_file function
                trace_filtered = clean_sac_file(input_file, output_file)
                trace_filtered.plot()
                # trace_filtered.write(output_file, format='SAC')
            
                ix = input("Press Enter to continue...")

                if ix == 'q':
                    break




In [ ]:
import glob

directory = './'  # Replace with the actual directory path you want to iterate over

# Find all items (files and directories) in the directory
items = glob.glob(directory + '/*')

# Iterate over the items
for item in items:
    if os.path.isfile(item):  # Check if the item is a file
        print(f"File: {os.path.basename(item)}")
    elif os.path.isdir(item):  # Check if the item is a directory
        print(f"Directory: {os.path.basename(item)}")
    else:
        print(f"Other: {os.path.basename(item)}")


In [ ]:

# #  read sac file


# from obspy import read, Trace, Stream

# path = "./"
# folder_class = "class6"

# for j, file in enumerate(os.listdir(path= path + folder_class)):
#     print(file)
#     trace = read(path + folder_class + "/" + file)[0]
#     trace.plot()
#     ix = input("Press Enter to continue...")
#     if ix == 'q':
#         break



